# Import Dependencies

In [ ]:
import pymupdf as pmp
import re

print(pmp.__doc__)

In [ ]:
%pwd

In [ ]:
import os
os.chdir("./..")
%pwd

In [ ]:
#docs = pmp.open("/home/jovyan/work/EU_MDR_RAG/data/raw/eu_mdr_2017-745.pdf")
docs = pmp.open("data/raw/eu_mdr_2017-745.pdf")
print(len(docs))

# How Textual information appears

In [ ]:
page_1 = docs[0].get_text()
page_1

In [ ]:
page_2 = docs[1].get_text()
page_2

In [ ]:
page_140 = docs[139].get_text()
page_140

# How Tables appears?

In [ ]:
page_174 = docs[174].find_tables()
page_174

In [ ]:
len(page_174.tables)

In [ ]:
page_174.tables[0]

In [ ]:
page_174.tables[0].to_pandas()

In [ ]:
page_174.tables[0].to_markdown()

# How titles and Sections appears?

In [ ]:
page_13 = docs[12].get_text()
page_13

In [ ]:
page_15 = docs[14].get_text()
page_15

In [ ]:
page_94 = docs[93].get_text()
page_94

In [ ]:
page_71 = docs[70].get_text()
page_71

## Observations:
1. All chapters start on new lines, and the chapter name appears in capital letters on a new line.
2. Article numbers are in continuation it does not reset with chapters and this is not an issue for extraction.
3. All article numbers start on a new line, and article names follow immediately after the article number.
4. Some chapters have various numbered sections with names, and the articles reside under those sections.
5. Annexes also follow the same structure, but the chapters reside inside the Annexes.

In [ ]:
import re
pattern_annex = "^ANNEX [IVX]+"
re.findall(pattern_annex, page_94, re.M)

In [ ]:
# Get all the text
doc = pmp.open("/home/jovyan/work/EU_MDR_RAG/data/raw/eu_mdr_2017-745.pdf")
data = []
for page in doc:
    text = page.get_text()
    x = re.findall(pattern_annex, text, re.M)
    data.append(x) if not len(x)==0 else ""
    

In [ ]:
doc = pmp.open("data/raw/eu_mdr_2017-745.pdf")
page_starts = []
data=""
for num,page in enumerate(doc):
    page_starts.append(len(data))
    if num >= 173:
        break
    text = str(page.get_text())
    data += text

In [ ]:
data

In [ ]:
page_starts

In [ ]:
re.findall(pattern_annex, data, re.M)

In [ ]:
import re
pattern_annex = "^(ANNEX [IVX]+) \n(.+)"
x =(re.match(pattern_annex, "ANNEX I \nGENERAL SAFETY AND PERFORMANCE REQUIREMENTS", re.M)).group()
x

In [ ]:
import re
pattern_annex = "^(ANNEX [IVX]+) \n(.+)"
re.findall(pattern_annex, data, re.M)

# Observations:
1. Some ANNEX titles are cut when the new line comes.
2. When observed thr original title in the document, Another Part or section starts with capital letters so need to find a way to extract them.

In [ ]:
page_115 = docs[114].get_text()
page_115

# Observation
1. This is a tricky edge case. Capturing multi-line title is hard to get as it has the same pattern as header and sections.
2. We are doing this for citation purpose and for citation it is not important to capture the header completely. Mostly ANNEX number is sufficient enough.

In [ ]:
import re
pattern_chapter = "^(CHAPTER [IVX]+) \n(.+)"
re.findall(pattern_chapter, data, re.M)

In [ ]:
import re
pattern_article = "^(Article [0-9]+) \n(?!Article)(?!— )(.+)"
len(re.findall(pattern_article, data, re.M))

In [ ]:
import re
pattern_section = "^(SECTION [0-9]+) \n(.+)"
re.findall(pattern_section, data, re.M)

# Observations:
In the main body of the regulation:

Chapters contain articles
Article numbers never reset (1 through 123)
New chapter doesn't reset anything — articles just keep incrementing
Sections appear inside some chapters but articles still keep their global numbering

In the annexes:

Annexes contain chapters
Chapter numbers reset within each annex (each annex starts with Chapter I again)
But annexes don't contain articles — they have numbered items like "1.", "1.1.", "2.3." instead.", "2.3." instead

In [ ]:
a = """abcdefghi
I am There"""
if 'a' in a:
    print("A is there")
elif 'b' in a:
    print("B is there")
elif 'c' in a:
    print("C is there")
else:
    print("No one is there")

In [ ]:
Documents = []
current_text = ""
chapter = ""
section = ""
section_header = ""
metadata = {"document_name":,
"document_type":,
"chapter":,
"article_number": ,
"article_title":,
"annex": ,"section_header":,"paragraph_number": ,"page_number": ,"cross_references": }

for each line:
    if line matches ANNEX pattern:
        annex = matched pattern
        metadata.update(fill the annex in metadata and page number document_name and type leave the other fields as none)
        if current_text is not empty:
            doc = Document(page_content=current_text, metadata = metadata)
        chapter = ""
        current_text = ""
        
    elif line matches CHAPTER pattern:
        metadata.update(fill the chapter, page_number and document type and name)
        doc = Document(page_content=current_text, metadata = metadata)
        section = ""
        current_text = ""
    elif line matches SECTION pattern:
        metadata.update(fill the section, page_number and document type and name)
        doc = Document(page_content=current_text, metadata = metadata)
        section = ""
        current_text = ""
        
    elif line matches Article pattern:
        metadata.update(fill the article, page_number and document type and name)
    else:
        current_text

### Writing a Pseudocode for structure Parser

In [ ]:
Documents = []
current_text = ""
chapter = ""
section = ""
section_header = ""
line = 0
metadata = {"document_name":"eu_mdr_2017-745.pdf","document_type":"Regulations","chapter":None,"article_number":None ,"article_title":None,"annex":None ,"section_header":None,"paragraph_number": None,"page_number": None,"cross_references": None}

for page_num, page in enumerate(documents):
    for line in page:
        new_metadata = metadata.copy()
        if line matches with ANNEX pattern:
            if current_text is not None:
                page_content = current_text
                Documents.append(Document(page_content, new_metadata))
            annex = matched pattern
            new_metadata['annex']=annex
            new_metadata['page_number'] = page_num
            metadata.update(new_metadata)
            current_text = None
            if chapter is not None:    
                chapter = None
            
        elif line matches with chapter pattern:
            if current_text is not None:
                    page_content = current_text
                    Documents.append(Document(page_content, new_metadata))
            chapter = matched pattern
            new_metadata['chapter']=chapter
            new_metadata['page_number'] = page_num
            metadata.update(new_metadata)
            current_text = None
            if section is not None:    
                section = None
                
        elif line machtes with section pattern:
            if current_text is not None:
                page_content = current_text
                Documents.append(Document(page_content, new_metadata))
            section = matched pattern
            new_metadata['section']=section
            new_metadata['page_number'] = page_num
            metadata.update(new_metadata)
            current_text = None
            
        elif line matches with article pattern:
            if current_text is not None:
                page_content = current_text
                Documents.append(Document(page_content, new_metadata))
            article = matched pattern
            new_metadata['article_number'],new_metadata['article_title'] = article
             = article.split(" ")[1]
            new_metadata['page_number'] = page_num
            metadata.update(new_metadata)
            current_text = None
        else:
            current_text += line

## Converting into python

In [ ]:
Docs = pmp.open("C:\\Users\\Vraj\\Desktop\\EU_MDR_RAG\\data\\raw\\eu_mdr_2017-745.pdf")
docs = Docs[93].get_text()
docs

In [ ]:
Documents = []
current_text = ""
chapter = ""
section = ""
section_header = ""
pattern_annex = "^(ANNEX [IVX]+)"# \n(.+)"
pattern_chapter = "^(CHAPTER [IVX]+)"# \n(.+)"
pattern_section = "^(SECTION [0-9]+)"# \n(.+)"
pattern_article = "^(Article [0-9]+)"# \n(?!Article)(?!— )(.+)"
metadata = {"document_name": "eu_mdr_2017-745.pdf", "document_type": "Regulations", "chapter": "", "article_number": "", "article_title": "", "annex": "", "section": "", "paragraph_number": "", "page_number": "", "cross_references": ""}

for page_num, page in enumerate(Docs):
    for line in (page.get_text()).split("\n"):
        new_metadata = metadata.copy()
        if re.match(pattern_annex, line):
            if current_text:
                page_content = current_text
                Documents.append({"page_content":page_content, "metadata":new_metadata})
            annex = re.match(pattern_annex, line, re.M).group()
            new_metadata["annex"] = annex
            new_metadata["page_number"] = page_num + 1
            metadata.update(new_metadata)
            current_text = ""
            if chapter != "":    
                chapter = ""
        
        elif re.match(pattern_chapter, line):
            if current_text:
                page_content = current_text
                Documents.append({"page_content":page_content, "metadata": new_metadata})
            chapter = re.match(pattern_chapter, line, re.M).group()
            new_metadata['chapter'] = chapter
            new_metadata["page_number"] = page_num + 1
            metadata.update(new_metadata)
            current_text = ""
            if section != "":
                section = ""
        
        elif re.match(pattern_section, line):
            if current_text:
                page_content = current_text
                Documents.append(({"page_content":page_content, "metadata": new_metadata}))
            section =  re.match(pattern_section, line, re.M)
            new_metadata['section'] = section
            new_metadata["page_number"] = page_num + 1
            metadata.update(new_metadata)
            current_text = ""
        
        elif re.match(pattern_article, line):
            if current_text:
                page_content = current_text
                Documents.append(({"page_content":page_content, "metadata": new_metadata}))
            article =  re.match(pattern_article, line, re.M)
            new_metadata['article_number'] = article
            new_metadata["page_number"] = page_num + 1
            metadata.update(new_metadata)
            current_text = ""
        
        else:
            current_text += "\n" + line
    
if current_text:
        Documents.append({"page_content": current_text, "metadata": metadata.copy()})

for doc in Documents:
    print(doc)

### Observation:
1. Here, in the regulations the new line starts with Article 8, which pattern matches and creates a new document. The issue is it is not new article and will lead to poor results.
2. Also, i altered the pattern because the regex matches with only one line so i need to either find the all the patterns and then extract the text between them or i have to create two line lookahead function.

In [ ]:
import re
pattern_chapter = "^(CHAPTER [IVX]+) \n(.+)"
s = re.finditer(pattern_chapter, data, re.M)
for m in s:
    print(m.group())

### Idea:
The plan is to collect all the title's location and extract the data between them.

In [ ]:
pattern_annex = "^(ANNEX [IVX]+) \n(.+)"
pattern_chapter = "^(CHAPTER [IVX]+) \n(.+)"
pattern_section = "^(SECTION [0-9]+) \n(.+)"
pattern_article ="^(Article [0-9]+) \n(?!Article)(?!— )(.+)"
markers = list()
for pattern in [pattern_annex, pattern_article, pattern_chapter, pattern_section]:
    iter_obj = re.finditer(pattern, data, re.M)
    for match in iter_obj:
        if pattern == pattern_annex:
            markers.append((match.start(),match.end(), "Annex", match.group()))
        if pattern == pattern_chapter:
            markers.append((match.start(),match.end(), "Chapter", match.group()))
        if pattern == pattern_article:
            markers.append((match.start(),match.end(), "Article", match.group()))
        if pattern == pattern_section:
            markers.append((match.start(),match.end(), "Section", match.group()))
sorted_markers=sorted(markers)
sorted_markers

In [ ]:
len(sorted_markers)

In [ ]:
import bisect
p = bisect.bisect(page_starts,sorted_markers[25][0])
p

In [ ]:
import bisect
Documents = []
current_text = ""
chapter = ""
section = ""
section_header = ""
start_location = 0
end_location = sorted_markers[-1][1]
metadata = {"document_name": "eu_mdr_2017-745.pdf", "document_type": "Regulations", "chapter": "", "chapter_title": "", "article": "", "article_title": "", "annex": "","annex_title": "", "section": "", "section_title": "", "page_number": "", "cross_references": ""}

pc = data[0:sorted_markers[0][1]]
mc = metadata.copy()
mc['chapter'] = 0
mc['chapter_title'] = 'preamble'
mc['page_number'] = bisect.bisect(page_starts, sorted_markers[0][0]-1)
doc_1 =Documents.append({"page_content": pc, "metadata": mc})

for i, marker in enumerate(sorted_markers):
    if i < len(sorted_markers) - 1:
        current_text = data[marker[0]:sorted_markers[i+1][0]]
    else:
        current_text = data[marker[1]:]
    new_metadata = metadata.copy()

    if marker[2] == "Chapter":
        chapter = marker[3].split("\n")[0]
        chapter_title = marker[3].split("\n")[1]
        new_metadata['chapter'] = chapter
        new_metadata['chapter_title'] = chapter_title
        ch_location = marker[0]
        page_num = bisect.bisect(page_starts,ch_location)
        new_metadata['page_number'] = page_num
        metadata.update(new_metadata)
        Documents.append({"page_content": current_text, "metadata": new_metadata})

    elif marker[2] == "Section":
        section = marker[3].split("\n")[0]
        section_title = marker[3].split("\n")[1]
        new_metadata['section'] = section
        new_metadata['section_title'] = section_title
        sec_location = marker[0]
        page_num = bisect.bisect(page_starts,sec_location)
        new_metadata['page_number'] = page_num
        metadata.update(new_metadata)
        Documents.append({"page_content": current_text, "metadata": new_metadata})
    
    elif marker[2] == "Article":
        article = marker[3].split("\n")[0]
        article_title = marker[3].split("\n")[1]
        new_metadata['article'] = article
        new_metadata['article_title'] = article_title
        art_location = marker[0]
        page_num = bisect.bisect(page_starts,art_location)
        new_metadata['page_number'] = page_num
        metadata.update(new_metadata)
        Documents.append({"page_content": current_text, "metadata": new_metadata})

    
    elif marker[2] == "Annex":
        annex = marker[3].split("\n")[0]
        annex_title = marker[3].split("\n")[1]
        new_metadata['annex'] = annex
        new_metadata['annex_title'] = annex_title
        ann_location = marker[0]
        page_num = bisect.bisect(page_starts,ann_location)
        new_metadata['page_number'] = page_num
        metadata.update(new_metadata)
        Documents.append({"page_content": current_text, "metadata": new_metadata})
        metadata['chapter'] = ""
        metadata['chapter_title'] = ""
        metadata['article'] = ""
        metadata['article_title'] = ""
        metadata['section'] = ""
        metadata['section_title'] = ""
            
    #Documents.append({"page_content": current_text, "metadata": metadata})

In [ ]:
len(Documents)

In [ ]:
sorted(Documents, key = lambda x:x['metadata'].get('page_number',0))

Observation:
1. Now, all the data extratced and created as document with metadata.
2. I still need to do data cleaning like removing footers and so.
3. I observed the pattern by checking the documents using pymupdf.open. I checked first three pages and some random pages how the headers and footers are represented.
4. The pattern is simillar almost at every page. Theheader stats with:

`" \n5.5.2017 \nL 117/1 \n.............."
`
The footer starts in the same way butwith extra spaces like shown below:

`"     \n(1) Regulation (EC) No 178/2002 of the European Parliament and of"`

almost at every page.

* But first remove the uncode characters for consistency.

Hint:
`ord()` function

In [ ]:
special_chr = []
d = {}
for i in data:
    p = ord(i)
    if p > 127:
        special_chr.append(chr(p))
        d[p]=chr(p)

print(set(special_chr),d)

In [ ]:
data[data.find("Ε")-50: data.find("Ε")+50]

In [ ]:
data[data.find("à")-50: data.find("à")+50]

In [ ]:
data.find("\xad")

In [ ]:
data[data.find("\xad")-50: data.find("\xad")+50]

In [ ]:
print(data.count("à"), data.count("\xad"), data.count("‑"), data.count('‘'), data.count('’'), data.count('Ε'))

In [ ]:
def clean(corpus):

    corpus = corpus.replace("\xad","")
    corpus = corpus.replace("Ε","E")
    corpus = corpus.replace("‑","-")
    corpus = corpus.replace("‘","'")
    corpus = corpus.replace("’","'")

    return corpus

new_data = clean(data)

In [ ]:
print(new_data.count("à"), new_data.count("\xad"), new_data.count("‑"), new_data.count('‘'), new_data.count('’'), new_data.count('Ε'))

In [ ]:
new_data[4250:4350]

### Now, removeing Header/footers

In [ ]:
header_pattern = r"""\d{1,2}\.\d{1,2}\.\d{4}\s*\nL\s+\d{3}/\d+\s*\nOfficial Journal of the European Union\s*\nEN\s*"""
re.findall(header_pattern, new_data)

Observation:
1. One thing to see is the footer contains the cross reference with some stated papers as follows:

`Opinion of 14 February 2013 (OJ C 133, 9.5.2013, p. 52).`

for moment i am planning to keep it. if in future i make an agent for it then it might help to find the relation.

In [ ]:
# Remove Headers

def remove_header(corpus):
    new_data=re.sub(header_pattern,"",corpus)